# Code as Policies Sagittarius Real-Robot Interactive Demo

This notebook is a slimmed-down real-robot variant of the Code as Policies interactive demo.

It keeps the full LMP stack and uses `SagittariusEnv` to bridge high-level generated code to the `sagittarius_ws` backend.

**Instructions:**

1. Start the Sagittarius ROS backend and camera before running the interactive cells.
2. Run the setup and LMP cells in order.
3. Enter your OpenAI API key in the dedicated prompt cell.
4. Initialize `SagittariusEnv`, confirm the detected objects, then run commands from the final interactive cell.


In [1]:
model_name = 'gpt-3.5-turbo-instruct'  # 'text-davinci-002'

import importlib
import importlib.metadata
import subprocess
import sys


def ensure_package(spec, import_name=None, distribution_name=None, version=None):
    import_name = import_name or spec.split('==')[0].split('[')[0].replace('-', '_')
    distribution_name = distribution_name or spec.split('==')[0].split('[')[0]

    needs_install = False
    try:
        importlib.import_module(import_name)
        if version is not None:
            installed_version = importlib.metadata.version(distribution_name)
            needs_install = installed_version != version
    except Exception:
        needs_install = True

    if not needs_install:
        return

    print(f'Installing {spec} into {sys.executable}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', spec])
    importlib.invalidate_caches()
    importlib.import_module(import_name)


required_packages = [
    dict(spec='numpy', import_name='numpy'),
    dict(spec='scipy', import_name='scipy'),
    dict(spec='shapely', import_name='shapely'),
    dict(spec='astunparse', import_name='astunparse'),
    dict(spec='pygments', import_name='pygments'),
    dict(spec='opencv-python', import_name='cv2', distribution_name='opencv-python'),
    dict(spec='openai==0.28.1', import_name='openai', distribution_name='openai', version='0.28.1'),
]

for pkg in required_packages:
    ensure_package(**pkg)

import ast
import copy
from time import sleep

import astunparse
import cv2
import numpy as np
import openai
import shapely
from openai.error import APIConnectionError, RateLimitError
from pygments import highlight
from pygments.formatters import TerminalFormatter
from pygments.lexers import PythonLexer
from shapely.affinity import *
from shapely.geometry import *


In [2]:
from shapely.geometry import *
from shapely.affinity import *

In [3]:
import ast
import astunparse
import copy
from time import sleep 
import numpy as np 
import openai 
import shapely
from openai.error import RateLimitError, APIConnectionError
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import TerminalFormatter

In [4]:
class LMP:

    def __init__(self, name, cfg, lmp_fgen, fixed_vars, variable_vars):
        # Store runtime config and shared registries for this LMP instance.
        # 保存该 LMP 实例的运行配置与共享变量注册表。
        self._name = name
        self._cfg = cfg

        self._base_prompt = self._cfg['prompt_text']

        self._stop_tokens = list(self._cfg['stop'])

        self._lmp_fgen = lmp_fgen

        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars
        self.exec_hist = ''

    def clear_exec_hist(self):
        # Reset session execution history.
        # 清空会话执行历史。
        self.exec_hist = ''

    def build_prompt(self, query, context=''):
        # Expose runtime helper APIs via a prompt placeholder.
        # 通过 prompt 占位符向模型暴露运行时可用 API。
        if len(self._variable_vars) > 0:
            variable_vars_imports_str = f"from utils import {', '.join(self._variable_vars.keys())}"
        else:
            variable_vars_imports_str = ''
        prompt = self._base_prompt.replace('{variable_vars_imports}', variable_vars_imports_str)

        # Append previous execution history for multi-turn continuity (optional).
        # 可选地拼接历史执行代码，支持多轮连续上下文。
        if self._cfg['maintain_session']:
            prompt += f'\n{self.exec_hist}'

        # Append world state (e.g., current objects) for grounded planning.
        # 追加世界状态（如当前物体），让模型基于环境生成动作。
        if context != '':
            prompt += f'\n{context}'

        # Format the user instruction with task-specific prefix/suffix.
        # 使用任务专用前后缀包装用户指令。
        use_query = f'{self._cfg["query_prefix"]}{query}{self._cfg["query_suffix"]}'
        prompt += f'\n{use_query}'

        return prompt, use_query

    def __call__(self, query, context='', **kwargs):
        # 1) Build prompt from template + history + context + query.
        # 第1步：由模板、历史、上下文与当前指令构造 prompt。
        prompt, use_query = self.build_prompt(query, context=context)

        # 2) Ask LLM for executable Python policy code (retry on transient API errors).
        # 第2步：向 LLM 请求可执行策略代码（临时 API 错误会重试）。
        while True:
            try:
                code_str = openai.Completion.create(
                    prompt=prompt,
                    stop=self._stop_tokens,
                    temperature=self._cfg['temperature'],
                    engine=self._cfg['engine'],
                    max_tokens=self._cfg['max_tokens']
                )['choices'][0]['text'].strip()
                break
            except (RateLimitError, APIConnectionError) as e:
                print(f'OpenAI API got err {e}')
                print('Retrying after 10s.')
                sleep(10)

        # 3) Prepare execution/log text; optionally prepend context code.
        # 第3步：准备执行文本与日志文本；可选地把 context 作为代码前缀。
        if self._cfg['include_context'] and context != '':
            to_exec = f'{context}\n{code_str}'
            to_log = f'{context}\n{use_query}\n{code_str}'
        else:
            to_exec = code_str
            to_log = f'{use_query}\n{to_exec}'

        to_log_pretty = highlight(to_log, PythonLexer(), TerminalFormatter())
        print(f'LMP {self._name} exec:\n\n{to_log_pretty}\n')

        # 4) Auto-generate missing helper functions referenced in generated code.
        # 第4步：自动补齐生成代码中引用但缺失的辅助函数。
        new_fs = self._lmp_fgen.create_new_fs_from_code(code_str)
        self._variable_vars.update(new_fs)

        # 5) Build execution scope and run generated code (unless in debug mode).
        # 第5步：构建执行作用域并运行代码（debug 模式下不执行）。
        gvars = merge_dicts([self._fixed_vars, self._variable_vars])
        lvars = kwargs

        if not self._cfg['debug_mode']:
            exec_safe(to_exec, gvars, lvars)

        # 6) Persist history and session variables for future turns.
        # 第6步：持久化执行历史与会话变量，供后续轮次复用。
        self.exec_hist += f'\n{to_exec}'

        if self._cfg['maintain_session']:
            self._variable_vars.update(lvars)

        if self._cfg['has_return']:
            return lvars[self._cfg['return_val_name']]


class LMPFGen:

    def __init__(self, cfg, fixed_vars, variable_vars):
        # Initialize function generator with prompt/config and current symbol tables.
        # 使用提示模板/配置与当前符号表初始化函数生成器。
        self._cfg = cfg

        self._stop_tokens = list(self._cfg['stop'])
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars

        self._base_prompt = self._cfg['prompt_text']

    def create_f_from_sig(self, f_name, f_sig, other_vars=None, fix_bugs=False, return_src=False):
        # Generate one function implementation from a function signature.
        # 根据函数签名生成对应实现。
        print(f'Creating function: {f_sig}')

        # Build a signature-focused query and call LLM for function source code.
        # 构造签名导向查询并调用 LLM 生成函数源码。
        use_query = f'{self._cfg["query_prefix"]}{f_sig}{self._cfg["query_suffix"]}'
        prompt = f'{self._base_prompt}\n{use_query}'

        while True:
            try:
                f_src = openai.Completion.create(
                    prompt=prompt, 
                    stop=self._stop_tokens,
                    temperature=self._cfg['temperature'],
                    engine=self._cfg['engine'],
                    max_tokens=self._cfg['max_tokens']
                )['choices'][0]['text'].strip()
                break
            except (RateLimitError, APIConnectionError) as e:
                print(f'OpenAI API got err {e}')
                print('Retrying after 10s.')
                sleep(10)

        # Optional post-edit pass for syntax/readability fixes.
        # 可选的二次编辑：修复语法并提升可读性。
        if fix_bugs:
            f_src = openai.Edit.create(
                model='code-davinci-edit-001',
                input='# ' + f_src,
                temperature=0,
                instruction='Fix the bug if there is one. Improve readability. Keep same inputs and outputs. Only small changes. No comments.',
            )['choices'][0]['text'].strip()

        # Execute generated source in a controlled scope and fetch callable.
        # 在受控作用域执行生成源码并取出可调用函数。
        if other_vars is None:
            other_vars = {}
        gvars = merge_dicts([self._fixed_vars, self._variable_vars, other_vars])
        lvars = {}
        
        exec_safe(f_src, gvars, lvars)

        f = lvars[f_name]

        # Pretty-print generated function for debugging/inspection.
        # 高亮打印生成函数，便于调试与审阅。
        to_print = highlight(f'{use_query}\n{f_src}', PythonLexer(), TerminalFormatter())
        print(f'LMP FGEN created:\n\n{to_print}\n')

        if return_src:
            return f, f_src
        return f

    def create_new_fs_from_code(self, code_str, other_vars=None, fix_bugs=False, return_src=False):
        # Parse generated code and find referenced function calls/assignments.
        # 解析生成代码，提取其中引用的函数调用与赋值。
        fs, f_assigns = {}, {}
        f_parser = FunctionParser(fs, f_assigns)
        f_parser.visit(ast.parse(code_str))
        for f_name, f_assign in f_assigns.items():
            if f_name in fs:
                fs[f_name] = f_assign

        # Initialize recursion state holders.
        # 初始化递归过程中的状态容器。
        if other_vars is None:
            other_vars = {}

        new_fs = {}
        srcs = {}
        for f_name, f_sig in fs.items():
            all_vars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
            # Only generate functions that do not already exist in scope.
            # 仅为当前作用域中尚不存在的函数进行生成。
            if not var_exists(f_name, all_vars):
                f, f_src = self.create_f_from_sig(f_name, f_sig, new_fs, fix_bugs=fix_bugs, return_src=True)

                # Recursively define child functions found in generated parent function body.
                # 若父函数体中出现子函数调用，则递归补齐子函数。
                f_def_body = astunparse.unparse(ast.parse(f_src).body[0].body)
                child_fs, child_f_srcs = self.create_new_fs_from_code(
                    f_def_body, other_vars=all_vars, fix_bugs=fix_bugs, return_src=True
                )

                if len(child_fs) > 0:
                    new_fs.update(child_fs)
                    srcs.update(child_f_srcs)

                    # Rebind parent function so newly created children are visible in scope.
                    # 重新绑定父函数，使新生成子函数在其作用域内可见。
                    gvars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
                    lvars = {}
                    
                    exec_safe(f_src, gvars, lvars)
                    
                    f = lvars[f_name]

                new_fs[f_name], srcs[f_name] = f, f_src

        if return_src:
            return new_fs, srcs
        return new_fs


class FunctionParser(ast.NodeTransformer):
    # AST visitor that collects function call signatures from generated code.
    # 用于从生成代码 AST 中提取函数调用签名的访问器。

    def __init__(self, fs, f_assigns):
      super().__init__()
      self._fs = fs
      self._f_assigns = f_assigns

    def visit_Call(self, node):
        # Record call signature, e.g., foo(a=1).
        # 记录函数调用签名，例如 foo(a=1)。
        self.generic_visit(node)
        if isinstance(node.func, ast.Name):
            f_sig = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.func).strip()
            self._fs[f_name] = f_sig
        return node

    def visit_Assign(self, node):
        # Track assignment-style calls to keep richer call context.
        # 跟踪赋值形式的函数调用，保留更完整上下文。
        self.generic_visit(node)
        if isinstance(node.value, ast.Call):
            assign_str = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.value.func).strip()
            self._f_assigns[f_name] = assign_str
        return node


def var_exists(name, all_vars):
    # Check whether a symbol already exists in current scope dictionary.
    # 检查符号是否已存在于当前作用域字典中。
    try:
        eval(name, all_vars)
    except:
        exists = False
    else:
        exists = True
    return exists


def merge_dicts(dicts):
    # Merge a list of dicts into one flat dict (later entries override earlier ones).
    # 将多个字典合并为一个（后者覆盖前者）。
    return {
        k : v 
        for d in dicts
        for k, v in d.items()
    }
    

def exec_safe(code_str, gvars=None, lvars=None):
    # Minimal safety guard before exec: block imports and dunder usage.
    # 执行前的最小安全保护：禁止 import 与双下划线用法。
    banned_phrases = ['import', '__']
    for phrase in banned_phrases:
        assert phrase not in code_str
  
    # Prepare default scopes and shadow dangerous built-ins.
    # 准备默认作用域并屏蔽高风险内置调用。
    if gvars is None:
        gvars = {}
    if lvars is None:
        lvars = {}
    empty_fn = lambda *args, **kwargs: None
    custom_gvars = merge_dicts([
        gvars,
        {'exec': empty_fn, 'eval': empty_fn}
    ])
    exec(code_str, custom_gvars, lvars)

In [5]:
# Global constants used by the LMP wrapper
COLORS = {
    'blue':   (78/255,  121/255, 167/255, 255/255),
    'red':    (255/255,  87/255,  89/255, 255/255),
    'green':  (89/255,  169/255,  79/255, 255/255),
    'orange': (242/255, 142/255,  43/255, 255/255),
    'yellow': (237/255, 201/255,  72/255, 255/255),
    'purple': (176/255, 122/255, 161/255, 255/255),
    'pink':   (255/255, 157/255, 167/255, 255/255),
    'cyan':   (118/255, 183/255, 178/255, 255/255),
    'brown':  (156/255, 117/255,  95/255, 255/255),
    'gray':   (186/255, 176/255, 172/255, 255/255),
}


## LMP Setup

### LMP Wrapper

In [6]:
class LMP_wrapper():

  def __init__(self, env, cfg, render=False):
    self.env = env
    self._cfg = cfg
    self.object_names = list(self.env.object_list)
    
    self._min_xy = np.array(self._cfg['env']['coords']['bottom_right'])
    self._max_xy = np.array(self._cfg['env']['coords']['top_left'])
    self._range_xy = self._max_xy - self._min_xy

    self._table_z = self._cfg['env']['coords']['table_z']
    self.render = render

  def is_obj_visible(self, obj_name):
    return obj_name in self.env.object_list

  def get_obj_names(self):
    return list(self.env.object_list)

  def denormalize_xy(self, pos_normalized):
    return pos_normalized * self._range_xy + self._min_xy

  #def get_corner_positions(self):
    #unit_square = box(0, 0, 1, 1)
    #normalized_corners = np.array(list(unit_square.exterior.coords))[:4]
    #corners = np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))
    #return corners

  #def get_side_positions(self):
    #side_xs = np.array([0, 0.5, 0.5, 1])
    #side_ys = np.array([0.5, 0, 1, 0.5])
    #normalized_side_positions = np.c_[side_xs, side_ys]
    #side_positions = np.array(([self.denormalize_xy(corner) for corner in normalized_side_positions]))
    #return side_positions

  def get_obj_pos(self, obj_name):
    # return the xy position of the object in robot base frame
    return self.env.get_obj_pos(obj_name)[:2]

  def get_obj_position_np(self, obj_name):
    return self.get_pos(obj_name)

  def get_bbox(self, obj_name):
    # return the axis-aligned object bounding box in robot base frame (not in pixels)
    # the format is (min_x, min_y, max_x, max_y)
    bbox = self.env.get_bounding_box(obj_name)
    return bbox

  def get_color(self, obj_name):
    for color, rgb in COLORS.items():
      if color in obj_name:
        return rgb

  def pick_place(self, pick_pos, place_pos):
    pick_pos_xyz = np.r_[pick_pos, [self._table_z]]
    place_pos_xyz = np.r_[place_pos, [self._table_z]]
    pass

  def put_first_on_second(self, arg1, arg2):
    # put the object with obj_name on top of target
    # target can either be another object name, or it can be an x-y position in robot base frame
    # Original env.step call kept for reference:
    # pick_pos = self.get_obj_pos(arg1) if isinstance(arg1, str) else arg1
    # place_pos = self.get_obj_pos(arg2) if isinstance(arg2, str) else arg2
    # result = self.env.step(action={'pick': pick_pos, 'place': place_pos})
    pick_pos = self.get_obj_pos(arg1) if isinstance(arg1, str) else arg1
    place_pos = self.get_obj_pos(arg2) if isinstance(arg2, str) else arg2
    object_name = arg1 if isinstance(arg1, str) else None
    target_name = arg2 if isinstance(arg2, str) else None
    result = self.env.step(
      action={
        'pick': pick_pos,
        'place': place_pos,
        'object_name': object_name,
        'target_name': target_name,
      }
    )
    if (not isinstance(result, dict)) or (not result.get('success', False)):
      reason = result.get('reason', 'unknown') if isinstance(result, dict) else 'invalid_result'
      print(f'put_first_on_second failed: {arg1} -> {arg2}, reason={reason}')
      return result
    return result

  def get_robot_pos(self):
    # return robot end-effector xy position in robot base frame
    return self.env.get_ee_pos()

  def goto_pos(self, position_xy, max_steps=60):
    # move the robot end-effector to the desired xy position while maintaining same z
    ee_xyz = self.env.get_ee_pos()
    position_xyz = np.concatenate([position_xy, [ee_xyz[-1]]])
    for _ in range(max_steps):
      if np.linalg.norm(position_xyz - ee_xyz) <= 0.01:
        return
      if not self.env.movep(position_xyz):
        raise RuntimeError(f'goto_pos failed: target={position_xyz.tolist()}')
      self.env.step_sim_and_render()
      ee_xyz = self.env.get_ee_pos()
    raise RuntimeError(
      f'goto_pos timed out: target={position_xyz.tolist()}, current={ee_xyz.tolist()}'
    )

  def follow_traj(self, traj):
    for pos in traj:
      self.goto_pos(pos)

  def get_corner_positions(self):
    normalized_corners = np.array([
        [1, 1], #top left
        [1, 0], #top right
        [0, 1], #bottom left
        [0, 0]  #bottom right
    ])
    return np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))

  def get_side_positions(self):
    normalized_sides = np.array([
        [1, 0.5], #top
        [0.5, 0], #right
        [0, 0.5], #bottom
        [0.5, 1]  #left
    ])
    return np.array(([self.denormalize_xy(side) for side in normalized_sides]))

  def get_corner_name(self, pos):
    corner_positions = self.get_corner_positions()
    corner_idx = np.argmin(np.linalg.norm(corner_positions - pos, axis=1))
    return ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner'][corner_idx]

  def get_side_name(self, pos):
    side_positions = self.get_side_positions()
    side_idx = np.argmin(np.linalg.norm(side_positions - pos, axis=1))
    return ['top side', 'right side', 'bottom side', 'left side'][side_idx]


### LMP Prompts

In [7]:
prompt_tabletop_ui = '''
# Python 2D robot control script
import numpy as np
from env_utils import put_first_on_second, get_obj_pos, get_obj_names, say, get_corner_name, get_side_name, is_obj_visible, stack_objects_in_order
from plan_utils import parse_obj_name, parse_position, parse_question, transform_shape_pts

objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# the yellow block on the yellow bowl.
say('Ok - putting the yellow block on the yellow bowl')
put_first_on_second('yellow block', 'yellow bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']

# move the green block to the top right corner.
say('Got it - putting the green block on the top right corner')
corner_pos = parse_position('top right corner')
put_first_on_second('green block', corner_pos)
objects = ['red block', 'green bowl', 'blue block', 'yellow bowl']
# put red block to area A.
say('Got it - putting the red block to area A')
area_pos = parse_position('area A')
put_first_on_second('red block', area_pos)
objects = ['green bowl', 'blue block', 'yellow bowl']
# move green bowl 10cm left.
say('Sure - moving the green bowl left by 10 centimeters')
left_pos = parse_position('a point 10cm left of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# stack the blue bowl on the yellow bowl on the green block.
order_bottom_to_top = ['green block', 'yellow block', 'blue bowl']
say(f'Sure - stacking from top to bottom: {", ".join(order_bottom_to_top)}')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# move the cyan block into its corresponding bowl.
matches = {'cyan block': 'cyan bowl'}
say('Got it - placing the cyan block on the cyan bowl')
for first, second in matches.items():
  put_first_on_second(first, get_obj_pos(second))
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# make a line of blocks on the right side.
say('No problem! Making a line of blocks on the right side')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
line_pts = parse_position(f'a 30cm vertical line on the right with {len(block_names)} points')
for block_name, pt in zip(block_names, line_pts):
  put_first_on_second(block_name, pt)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# make a 20cm horizontal line of blocks near the top side.
say('Ok - arranging the blocks in a 20cm horizontal line near the top side')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
line_pts = parse_position(f'a 20cm horizontal line near the top side with {len(block_names)} points')
for block_name, pt in zip(block_names, line_pts):
  put_first_on_second(block_name, pt)
# put the small banana colored thing in between the blue bowl and green block.
say('Sure thing - putting the yellow block between the blue bowl and the green block')
target_pos = parse_position('a point in the middle betweeen the blue bowl and the green block')
put_first_on_second('yellow block', target_pos)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# stack the blocks on the right side with the gray one on the bottom.
say('Ok. stacking the blocks on the right side with the gray block on the bottom')
right_side = parse_position('the right side')
put_first_on_second('gray block', right_side)
order_bottom_to_top = ['gray block', 'green block', 'yellow block']
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# stack everything with the green block on top.
say('Ok! Stacking everything with the green block on the top')
order_bottom_to_top = ['blue bowl', 'pink bowl', 'green bowl', 'pink block', 'blue block', 'green block']
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the grass-colored bowl to the left.
say('Sure - moving the green bowl left by 10 centimeters')
left_pos = parse_position('a point 10cm left of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# place the top most block to the corner closest to the bottom most block.
top_block_name = parse_obj_name('top most block', f'objects = {get_obj_names()}')
bottom_block_name = parse_obj_name('bottom most block', f'objects = {get_obj_names()}')
closest_corner_pos = parse_position(f'the corner closest to the {bottom_block_name}', f'objects = {get_obj_names()}')
say(f'Putting the {top_block_name} on the {get_corner_name(closest_corner_pos)}')
put_first_on_second(top_block_name, closest_corner_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the brown bowl to the side closest to the green block.
closest_side_position = parse_position('the side closest to the green block')
say(f'Got it - putting the brown bowl on the {get_side_name(closest_side_position)}')
put_first_on_second('brown bowl', closest_side_position)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the green block to the right of the bowl that has the blue block.
bowl_name = parse_obj_name('the bowl that has the blue block', f'objects = {get_obj_names()}')
if bowl_name:
  target_pos = parse_position(f'a point 10cm to the right of the {bowl_name}')
  say(f'No problem - placing the green block to the right of the {bowl_name}')
  put_first_on_second('green block', target_pos)
else:
  say('There are no bowls that has the blue block')
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the blue block in the empty bowl.
empty_bowl_name = parse_obj_name('the empty bowl', f'objects = {get_obj_names()}')
if empty_bowl_name:
  say(f'Ok! Putting the blue block on the {empty_bowl_name}')
  put_first_on_second('blue block', empty_bowl_name)
else:
  say('There are no empty bowls')
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the other blocks to the bottom corners.
block_names = parse_obj_name('blocks other than the blue block', f'objects = {get_obj_names()}')
corners = parse_position('the bottom corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the red bowl a lot to the left of the blocks.
say('Sure! Moving the red bowl to a point left of the blocks')
left_pos = parse_position('a point 20cm left of the blocks')
put_first_on_second('red bowl', left_pos)
objects = ['pink block', 'gray block', 'orange block']
# move the pinkish colored block on the bottom side.
say('Ok - putting the pink block on the bottom side')
bottom_side_pos = parse_position('the bottom side')
put_first_on_second('pink block', bottom_side_pos)
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# is the blue block to the right of the yellow bowl?
if parse_question('is the blue block to the right of the yellow bowl?', f'objects = {get_obj_names()}'):
  say('yes, there is a blue block to the right of the yellow bow')
else:
  say('no, there is\'t a blue block to the right of the yellow bow')
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# how many yellow objects are there?
n_yellow_objs = parse_question('how many yellow objects are there', f'objects = {get_obj_names()}')
say(f'there are {n_yellow_objs} yellow object')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the left most block to the green bowl.
left_block_name = parse_obj_name('left most block', f'objects = {get_obj_names()}')
say(f'Moving the {left_block_name} on the green bowl')
put_first_on_second(left_block_name, 'green bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the other blocks to different corners.
block_names = parse_obj_name(f'blocks other than the {left_block_name}', f'objects = {get_obj_names()}')
corners = parse_position('the corners')
say(f'Ok - moving the other {len(block_names)} blocks to different corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl.
if parse_question('is the pink block on the green bowl', f'objects = {get_obj_names()}'):
  say('Yes - the pink block is on the green bowl.')
else:
  say('No - the pink block is not on the green bowl.')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl.
left_block_names =  parse_question('what are the blocks left of the green bowl', f'objects = {get_obj_names()}')
if len(left_block_names) > 0:
  say(f'These blocks are left of the green bowl: {", ".join(left_block_names)}')
else:
  say('There are no blocks left of the green bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']

# which block is closest to the right side?
closest_block = parse_question('which block is closest to the right side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the right side')

# which block is closest to the left side?
closest_block = parse_question('which block is closest to the left side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the left side')

# which block is closest to the top side?
closest_block = parse_question('which block is closest to the top side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the top side')

# which block is closest to the bottom side?
closest_block = parse_question('which block is closest to the bottom side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the bottom side')

# move all blocks 5cm toward the top.
say('Ok - moving all blocks 5cm toward the top')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name in block_names:
  target_pos = parse_position(f'a point 5cm above the {block_name}')
  put_first_on_second(block_name, target_pos)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make a triangle of blocks in the middle.
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
triangle_pts = parse_position(f'a triangle with size 10cm around the middle with {len(block_names)} points')
say('Making a triangle of blocks around the middle of the workspace')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make the triangle smaller.
triangle_pts = transform_shape_pts('scale it by 0.5x', shape_pts=triangle_pts)
say('Making the triangle smaller')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['brown bowl', 'red block', 'brown block', 'red bowl', 'pink bowl', 'pink block']
# put the red block on the farthest bowl.
farthest_bowl_name = parse_obj_name('the bowl farthest from the red block', f'objects = {get_obj_names()}')
say(f'Putting the red block on the {farthest_bowl_name}')
put_first_on_second('red block', farthest_bowl_name)
'''.strip()

In [8]:
prompt_parse_obj_name = '''
import numpy as np
from env_utils import get_obj_pos, parse_position
from utils import get_obj_positions_np

objects = ['blue block', 'cyan block', 'purple bowl', 'gray bowl', 'brown bowl', 'pink block', 'purple block']
# the block closest to the purple bowl.
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the blocks.
ret_val = ['brown block', 'blue block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the brown objects.
ret_val = ['brown bowl', 'brown block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# a fruit that's not the apple
fruit_names = ['banana', 'apple']
for fruit_name in fruit_names:
    if fruit_name != 'apple':
        ret_val = fruit_name
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# blocks above the brown bowl.
block_names = ['blue block', 'cyan block', 'purple block']
brown_bowl_pos = get_obj_pos('brown bowl')
use_block_names = []
for block_name in block_names:
    if get_obj_pos(block_name)[1] > brown_bowl_pos[1]:
        use_block_names.append(block_name)
ret_val = use_block_names
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the blue block.
ret_val = 'blue block'
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the block closest to the bottom right corner.
corner_pos = parse_position('bottom right corner')
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=corner_pos)
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the left most block.
block_names = ['green block', 'brown block', 'blue block']
block_positions = get_obj_positions_np(block_names)
left_block_idx = np.argsort(block_positions[:, 0])[0]
left_block_name = block_names[left_block_idx]
ret_val = left_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the bowl on near the top.
bowl_names = ['brown bowl', 'green bowl', 'blue bowl']
bowl_positions = get_obj_positions_np(bowl_names)
top_bowl_idx = np.argsort(block_positions[:, 1])[-1]
top_bowl_name = bowl_names[top_bowl_idx]
ret_val = top_bowl_name
objects = ['yellow bowl', 'purple block', 'yellow block', 'purple bowl', 'pink bowl', 'pink block']
# the third bowl from the right.
bowl_names = ['yellow bowl', 'purple bowl', 'pink bowl']
bowl_positions = get_obj_positions_np(bowl_names)
bowl_idx = np.argsort(block_positions[:, 0])[-3]
bowl_name = bowl_names[bowl_idx]
ret_val = bowl_name
'''.strip()

In [9]:
prompt_parse_position = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *
from env_utils import denormalize_xy, parse_obj_name, get_obj_names, get_obj_pos

# a 30cm horizontal line in the middle with 3 points.
middle_pos = denormalize_xy([0.5, 0.5]) 
start_pos = middle_pos + [0, 0.3/2]
end_pos = middle_pos + [0, -0.3/2]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=3)
ret_val = points
# a 20cm vertical line near the right with 4 points.
middle_pos = denormalize_xy([0.5, 0]) 
start_pos = middle_pos + [-0.2/2, 0]
end_pos = middle_pos + [0.2/2, 0]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=4)
ret_val = points
# a 20cm horizontal line near the top side with 3 points.
middle_pos = get_obj_pos('top side')
start_pos = middle_pos + [0, 0.2/2]
end_pos = middle_pos + [0, -0.2/2]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=3)
ret_val = points
# a diagonal line from the top left to the bottom right corner with 5 points.
top_left_corner = denormalize_xy([1, 1])
bottom_right_corner = denormalize_xy([0, 0])
line = make_line(start=top_left_corner, end=bottom_right_corner)
points = interpolate_pts_on_line(line=line, n=5)
ret_val = points
# a triangle with size 10cm with 3 points.
polygon = make_triangle(size=0.1, center=denormalize_xy([0.5, 0.5]))
points = get_points_from_polygon(polygon)
ret_val = points
# the corner closest to the sun colored block.
block_name = parse_obj_name('the sun colored block', f'objects = {get_obj_names()}')
corner_positions = np.array([denormalize_xy(pos) for pos in [[1, 1], [1, 0], [0, 1], [0, 0]]])
closest_corner_pos = get_closest_point(points=corner_positions, point=get_obj_pos(block_name))
ret_val = closest_corner_pos
# the side farthest from the right most bowl.
bowl_name = parse_obj_name('the right most bowl', f'objects = {get_obj_names()}')
side_positions = np.array([denormalize_xy(pos) for pos in [[1, 0.5],[0.5, 0], [0, 0.5], [0.5, 1]]])
farthest_side_pos = get_farthest_point(points=side_positions, point=get_obj_pos(bowl_name))
ret_val = farthest_side_pos
# a point above the third block from the bottom.
block_name = parse_obj_name('the third block from the bottom', f'objects = {get_obj_names()}')
ret_val = get_obj_pos(block_name) + [0.1, 0]
# a point 10cm left of the bowls.
bowl_names = parse_obj_name('the bowls', f'objects = {get_obj_names()}')
bowl_positions = get_all_object_positions_np(obj_names=bowl_names)
left_obj_pos = bowl_positions[np.argmin(bowl_positions[:, 0])] + [-0.1, 0]
ret_val = left_obj_pos
# the bottom side.
bottom_pos = denormalize_xy([0, 0.5])
ret_val = bottom_pos
# the top side.
top_pos = denormalize_xy([1, 0.5])
ret_val = top_pos
# the top corners.
top_left_pos = denormalize_xy([1, 1])
top_right_pos = denormalize_xy([1, 0])
ret_val = [top_left_pos, top_right_pos]
# the right side
right_pos = denormalize_xy([0.5, 0])
# the left side
left_pos = denormalize_xy([0.5, 1])
# area A.
ret_val = get_obj_pos('area a')
# zone B.
ret_val = get_obj_pos('zone b')
# region C.
ret_val = get_obj_pos('region c')
# map area D.
ret_val = get_obj_pos('map area d')
'''.strip()

In [10]:
prompt_parse_question = '''
from utils import get_obj_pos, get_obj_names, parse_obj_name, bbox_contains_pt, is_obj_visible

objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# is the blue block to the right of the yellow bowl?
ret_val = get_obj_pos('blue block')[0] > get_obj_pos('yellow bowl')[0]
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# how many yellow objects are there?
yellow_object_names = parse_obj_name('the yellow objects', f'objects = {get_obj_names()}')
ret_val = len(yellow_object_names)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl?
ret_val = bbox_contains_pt(container_name='green bowl', obj_name='pink block')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
green_bowl_pos = get_obj_pos('green bowl')
left_block_names = []
for block_name in block_names:
  if get_obj_pos(block_name)[0] < green_bowl_pos[0]:
    left_block_names.append(block_name)
ret_val = left_block_names
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# which block is closest to the top side?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
top_side_x = get_obj_pos('top side')[0]
closest_block_name = block_names[0]
closest_dist = abs(get_obj_pos(closest_block_name)[0] - top_side_x)
for block_name in block_names[1:]:
  dist = abs(get_obj_pos(block_name)[0] - top_side_x)
  if dist < closest_dist:
    closest_block_name = block_name
    closest_dist = dist
ret_val = closest_block_name

# which block is closest to the bottom side?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
bottom_side_x = get_obj_pos('bottom side')[0]
closest_block_name = block_names[0]
closest_dist = abs(get_obj_pos(closest_block_name)[0] - bottom_side_x)
for block_name in block_names[1:]:
  dist = abs(get_obj_pos(block_name)[0] - bottom_side_x)
  if dist < closest_dist:
    closest_block_name = block_name
    closest_dist = dist
ret_val = closest_block_name

# which block is closest to the right side?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
right_side_y = get_obj_pos('right side')[1]
closest_block_name = block_names[0]
closest_dist = abs(get_obj_pos(closest_block_name)[1] - right_side_y)
for block_name in block_names[1:]:
  dist = abs(get_obj_pos(block_name)[1] - right_side_y)
  if dist < closest_dist:
    closest_block_name = block_name
    closest_dist = dist
ret_val = closest_block_name

# which block is closest to the left side?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
left_side_y = get_obj_pos('left side')[1]
closest_block_name = block_names[0]
closest_dist = abs(get_obj_pos(closest_block_name)[1] - left_side_y)
for block_name in block_names[1:]:
  dist = abs(get_obj_pos(block_name)[1] - left_side_y)
  if dist < closest_dist:
    closest_block_name = block_name
    closest_dist = dist
ret_val = closest_block_name

# is the sun colored block above the blue bowl?
sun_block_name = parse_obj_name('sun colored block', f'objects = {get_obj_names()}')
sun_block_pos = get_obj_pos(sun_block_name)
blue_bowl_pos = get_obj_pos('blue bowl')
ret_val = sun_block_pos[1] > blue_bowl_pos[1]
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the green block below the blue bowl?
ret_val = get_obj_pos('green block')[1] < get_obj_pos('blue bowl')[1]
'''.strip()

In [11]:
prompt_transform_shape_pts = '''
import numpy as np
from utils import get_obj_pos, get_obj_names, parse_position, parse_obj_name

# make it bigger by 1.5.
new_shape_pts = scale_pts_around_centroid_np(shape_pts, scale_x=1.5, scale_y=1.5)
# move it to the right by 10cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0.1, 0])
# move it to the top by 20cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0, 0.2])
# rotate it clockwise by 40 degrees.
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=-np.deg2rad(40))
# rotate by 30 degrees and make it slightly smaller
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=np.deg2rad(30))
new_shape_pts = scale_pts_around_centroid_np(new_shape_pts, scale_x=0.7, scale_y=0.7)
# move it toward the blue block.
block_name = parse_obj_name('the blue block', f'objects = {get_obj_names()}')
block_pos = get_obj_pos(block_name)
mean_delta = np.mean(block_pos - shape_pts, axis=1)
new_shape_pts = translate_pts_np(shape_pts, mean_delta)
'''.strip()

In [12]:
prompt_fgen = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *

from env_utils import get_obj_pos, get_obj_names
from ctrl_utils import put_first_on_second

# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)

# define function: y = eval_line(x, slope, y_intercept=0).
def eval_line(x, slope, y_intercept):
    return x * slope + y_intercept

# define function: pt = get_pt_to_the_left(pt, dist).
def get_pt_to_the_left(pt, dist):
    return pt + [-dist, 0]

# define function: pt = get_pt_to_the_top(pt, dist).
def get_pt_to_the_top(pt, dist):
    return pt + [0, dist]

# define function line = make_line_by_length(length=x).
def make_line_by_length(length):
  line = LineString([[0, 0], [length, 0]])
  return line

# define function: line = make_vertical_line_by_length(length=x).
def make_vertical_line_by_length(length):
  line = make_line_by_length(length)
  vertical_line = rotate(line, 90)
  return vertical_line

# define function: pt = interpolate_line(line, t=0.5).
def interpolate_line(line, t):
  pt = line.interpolate(t, normalized=True)
  return np.array(pt.coords[0])

# define function: pts = interpolate_pts_on_line(line, n=3).
def interpolate_pts_on_line(line, n):
  if n <= 0:
    return np.zeros((0, 2))
  if n == 1:
    return np.array([interpolate_line(line, 0.5)])
  ts = np.linspace(0.0, 1.0, n)
  pts = [interpolate_line(line, t) for t in ts]
  return np.array(pts)

# example: scale a line by 2.
line = make_line_by_length(1)
new_shape = scale(line, xfact=2, yfact=2)

# example: put object1 on top of object0.
put_first_on_second('object1', 'object0')

# example: get the position of the first object.
obj_names = get_obj_names()
pos_2d = get_obj_pos(obj_names[0])
'''.strip()

### LMP Config

In [13]:
cfg_tabletop = {
  'lmps': {
    'tabletop_ui': {
      'prompt_text': prompt_tabletop_ui,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': True,
      'debug_mode': False,
      'include_context': True,
      'has_return': False,
      'return_val_name': 'ret_val',
    },
    'parse_obj_name': {
      'prompt_text': prompt_parse_obj_name,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_position': {
      'prompt_text': prompt_parse_position,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_question': {
      'prompt_text': prompt_parse_question,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'transform_shape_pts': {
      'prompt_text': prompt_transform_shape_pts,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'new_shape_pts',
    },
    'fgen': {
      'prompt_text': prompt_fgen,
      'engine': model_name,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# define function: ',
      'query_suffix': '.',
      'stop': ['# define', '# example'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
    }
  }
}

lmp_tabletop_coords = {
        'top_left':     (0.35, 0.13),
        'top_side':     (0.35, 0.00),
        'top_right':    (0.35, -0.13),
        'left_side':    (0.25, 0.13),
        'middle':       (0.25, 0.00),
        'right_side':   (0.25, -0.13),
        'bottom_left':  (0.15, 0.13),
        'bottom_side':  (0.15, 0.00),
        'bottom_right': (0.15, -0.13),
        'table_z':       0.03,
      }

In [14]:
### LMP Utils

In [15]:
def interpolate_pts_on_line(line, n):
    if n <= 0:
        return np.zeros((0, 2))
    if n == 1:
        pt = line.interpolate(0.5, normalized=True)
        return np.array([pt.coords[0]])
    ts = np.linspace(0.0, 1.0, n)
    pts = [line.interpolate(float(t), normalized=True).coords[0] for t in ts]
    return np.array(pts)

def setup_LMP(env, cfg_tabletop):
  # LMP env wrapper
  cfg_tabletop = copy.deepcopy(cfg_tabletop)
  cfg_tabletop['env'] = dict()
  cfg_tabletop['env']['init_objs'] = list(env.obj_name_to_id.keys())
  cfg_tabletop['env']['coords'] = lmp_tabletop_coords
  LMP_env = LMP_wrapper(env, cfg_tabletop)
  # creating APIs that the LMPs can interact with
  fixed_vars = {
      'np': np,
      'interpolate_pts_on_line': interpolate_pts_on_line,
  }
  fixed_vars.update({
      name: eval(name)
      for name in shapely.geometry.__all__ + shapely.affinity.__all__
  })
  variable_vars = {
      k: getattr(LMP_env, k)
      for k in [
          'get_bbox', 'get_obj_pos', 'get_color', 'is_obj_visible', 'denormalize_xy',
          'put_first_on_second', 'get_obj_names',
          'get_corner_name', 'get_side_name',
      ]
  }
  variable_vars['say'] = lambda msg: print(f'robot says: {msg}')

  # creating the function-generating LMP
  lmp_fgen = LMPFGen(cfg_tabletop['lmps']['fgen'], fixed_vars, variable_vars)

  # creating other low-level LMPs
  variable_vars.update({
      k: LMP(k, cfg_tabletop['lmps'][k], lmp_fgen, fixed_vars, variable_vars)
      for k in ['parse_obj_name', 'parse_position', 'parse_question', 'transform_shape_pts']
  })

  # creating the LMP that deals w/ high-level language commands
  lmp_tabletop_ui = LMP(
      'tabletop_ui', cfg_tabletop['lmps']['tabletop_ui'], lmp_fgen, fixed_vars, variable_vars
  )

  return lmp_tabletop_ui

# Interactive Real-Robot Manipulation

Instructions:
1. Make sure `sagittarius_ws` backend nodes and the camera stream are already running.
2. Enter your OpenAI API key in the prompt cell below.
3. Run the environment initialization cell and confirm the detected object list.
4. Enter a command in the interactive cell and execute it.

Supported commands include spatial reasoning, sequential manipulation, contextual follow-ups, and simple Q&A over the currently detected objects.

Example commands:
* `put blue block into green bowl`
* `put red block to area A`
* `move green bowl 10cm left`
* `how many blocks are to the right of the blue bowl?`

Known limitations:
* Object positions are reconstructed from camera color segmentation and calibration, so lighting and calibration quality directly affect execution.
* The current real-robot path still uses simple pick/place primitives and does not perform rich object-state tracking after execution.
* Ambiguous instructions may still need to be rephrased into clearer object references.


In [16]:
#@title Initialize Env { vertical-output: true }
# --- SAGITTARIUS REAL ROBOT SETUP ---
import sys
sys.path.append('.')
from sagittarius_env import SagittariusEnv

env = SagittariusEnv(arm_name='sgr532')

SAFE_SCAN_RESET_POSE_CANDIDATES = [
    [0.20, 0.00, 0.10],
    [0.20, 0.00, 0.12],
    [0.20, 0.00, 0.15],
]

MAP_SCAN_POSES = [
    [0.20, 0.00, 0.10],
    [0.23, 0.09, 0.10],
    [0.26, -0.09, 0.10],
]

def reset_arm_before_map_scan():
    for pose in SAFE_SCAN_RESET_POSE_CANDIDATES:
        if env.movep(pose):
            return True
    try:
        cur = env.get_ee_pos().tolist()
    except Exception:
        cur = 'unknown'
    print(f'[WARN] reset_arm_before_map_scan failed. Continue with current pose: {cur}')
    return False

reset_arm_before_map_scan()
env.scan_world_state()
obj_list = env.object_list
lmp_tabletop_ui = setup_LMP(env, cfg_tabletop)

print('available objects:')
print(obj_list)


[INFO] [1777543749.906905]: Waiting for Sagittarius Control Action Server: /sgr532/sgr_ctrl
[INFO] [1777543749.921673]: Connected to Sagittarius control action server.
available objects:
['blue block', 'green block', 'red block']


In [17]:
import getpass
import openai

openai_api_key = getpass.getpass("OpenAI API key: ").strip()
openai.api_key = openai_api_key
print("API key loaded:", bool(openai.api_key))


OpenAI API key:  ········


API key loaded: True


In [20]:
#User input

user_input = 'which block is the closest to the right side, move it to top side, which block is the closest to the left side, move it to top right corner'

print('Running policy...')
lmp_tabletop_ui(user_input, f'objects = {env.object_list}')



Running policy...
LMP tabletop_ui exec:

objects = ['blue block', 'green block', 'red block']
# which block is the closest to the right side, move it to top side, which block is the closest to the left side, move it to top right corner.
closest_block = parse_question('which block is closest to the right side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the right side')
top_side = parse_position('top side')
put_first_on_second(closest_block, top_side)
closest_block = parse_question('which block is closest to the left side?', f'objects = {get_obj_names()}')
say(f'The {closest_block} is closest to the left side')
top_right_corner = parse_position('top right corner')
put_first_on_second(closest_block, top_right_corner)


LMP parse_question exec:

objects = ['blue block', 'green block', 'red block']
# which block is closest to the right side?.
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
right_side_y = get_obj_pos('right side')[1]


In [25]:
#print xy
import numpy as np

denorm = lmp_tabletop_ui._variable_vars['denormalize_xy']
wrapper = denorm.__self__

print("wrapper._min_xy =", wrapper._min_xy)
print("wrapper._max_xy =", wrapper._max_xy)
print("wrapper._range_xy =", wrapper._range_xy)

print("denormalize_xy([1,1]) =", wrapper.denormalize_xy(np.array([1, 1])))
print("denormalize_xy([1,0]) =", wrapper.denormalize_xy(np.array([1, 0])))
print("denormalize_xy([0,1]) =", wrapper.denormalize_xy(np.array([0, 1])))
print("denormalize_xy([0,0]) =", wrapper.denormalize_xy(np.array([0, 0])))

print("corner positions:")
print(wrapper.get_corner_positions())

print("side positions:")
print(wrapper.get_side_positions())

wrapper._min_xy = [ 0.15 -0.13]
wrapper._max_xy = [0.35 0.13]
wrapper._range_xy = [0.2  0.26]
denormalize_xy([1,1]) = [0.35 0.13]
denormalize_xy([1,0]) = [ 0.35 -0.13]
denormalize_xy([0,1]) = [0.15 0.13]
denormalize_xy([0,0]) = [ 0.15 -0.13]
corner positions:
[[ 0.35  0.13]
 [ 0.35 -0.13]
 [ 0.15  0.13]
 [ 0.15 -0.13]]
side positions:
[[ 0.35  0.  ]
 [ 0.25 -0.13]
 [ 0.15  0.  ]
 [ 0.25  0.13]]


import time
import numpy as np

MEASURE_Z = 0.03
COARSE_STEP = 0.01
FINE_STEP = 0.002

measured_points = {}

def ee_pos():
    return np.array(env.get_ee_pos(), dtype=float)

def fmt(v):
    return np.round(np.array(v, dtype=float).reshape(-1), 4).tolist()

def move_to_xyz(xyz):
    xyz = np.array(xyz, dtype=float)
    ok = env.movep(xyz)
    print("move_to", fmt(xyz), "->", ok)
    return ok

def set_measure_height(z=MEASURE_Z):
    cur = ee_pos()
    tgt = np.array([cur[0], cur[1], z], dtype=float)
    return move_to_xyz(tgt)

def jog(dx=0.0, dy=0.0, dz=0.0):
    cur = ee_pos()
    tgt = cur + np.array([dx, dy, dz], dtype=float)
    ok = env.movep(tgt)
    print("jog to", fmt(tgt), "->", ok)
    return ok

def record_point(name):
    measured_points[name] = ee_pos().copy()
    print(name, "=", fmt(measured_points[name]))

def show_points():
    for k, v in measured_points.items():
        print(k, "=", fmt(v))

In [42]:
set_measure_height()
print("current ee:", fmt(ee_pos()))

move_to [0.31, -0.09, 0.03] -> True
current ee: [0.31, -0.09, 0.03]


In [33]:
print("先确认 jog 的真实方向：")
print("依次手动执行下面命令，每次观察机械臂末端往哪边移动：")
print("jog(dx= COARSE_STEP)")
print("jog(dx=-COARSE_STEP)")
print("jog(dy= COARSE_STEP)")
print("jog(dy=-COARSE_STEP)")

print("\\n根据你的地图语义，确认：")
print("- 哪个方向是 top（远离机械臂）")
print("- 哪个方向是 bottom（靠近机械臂）")
print("- 哪个方向是 right（朝 A/C）")
print("- 哪个方向是 left（朝 B/D）")

先确认 jog 的真实方向：
依次手动执行下面命令，每次观察机械臂末端往哪边移动：
jog(dx= COARSE_STEP)
jog(dx=-COARSE_STEP)
jog(dy= COARSE_STEP)
jog(dy=-COARSE_STEP)
\n根据你的地图语义，确认：
- 哪个方向是 top（远离机械臂）
- 哪个方向是 bottom（靠近机械臂）
- 哪个方向是 right（朝 A/C）
- 哪个方向是 left（朝 B/D）


In [35]:
jog(dy= COARSE_STEP)

jog to [0.31, -0.09, 0.12] -> True


True

In [ ]:
print("""
现在开始测这些点：

中间大地图四角：
- top_left
- top_right
- bottom_left
- bottom_right

四个区域中心：
- area_a_center
- area_b_center
- area_c_center
- area_d_center

说明：
top = 远离机械臂
bottom = 靠近机械臂
right = 朝 A/C
left = 朝 B/D
""")

In [43]:
print("""
每个点都按这个流程测：

1. 用 jog(dx=...) / jog(dy=...) 粗调
2. 接近后用更小步长微调
3. 对准后执行 record_point("点名")

推荐顺序：
1. top_left
2. top_right
3. bottom_left
4. bottom_right
5. area_a_center
6. area_b_center
7. area_c_center
8. area_d_center
""")


每个点都按这个流程测：

1. 用 jog(dx=...) / jog(dy=...) 粗调
2. 接近后用更小步长微调
3. 对准后执行 record_point("点名")

推荐顺序：
1. top_left
2. top_right
3. bottom_left
4. bottom_right
5. area_a_center
6. area_b_center
7. area_c_center
8. area_d_center



In [101]:
jog(dy=-0.5)

jog to [0.26, -0.25, 0.03] -> True


True

In [102]:
record_point('area_c_center')

area_c_center = [0.26, -0.25, 0.03]


In [103]:
required = [
    "top_left",
    "top_right",
    "bottom_left",
    "bottom_right",
    "area_a_center",
    "area_b_center",
    "area_c_center",
    "area_d_center",
]

missing = [k for k in required if k not in measured_points]
if missing:
    print("还缺这些点：", missing)
else:
    print("点位已齐全。")
    show_points()

点位已齐全。
top_right = [0.35, -0.13, 0.03]
bottom_right = [0.15, -0.13, 0.03]
bottom_left = [0.15, 0.13, 0.03]
top_left = [0.35, 0.13, 0.03]
area_a_center = [0.16, -0.25, 0.03]
area_c_center = [0.26, -0.25, 0.03]
area_d_center = [0.26, 0.25, 0.03]
area_b_center = [0.16, 0.25, 0.03]


In [116]:
top_left = np.array(measured_points["top_left"], dtype=float)
top_right = np.array(measured_points["top_right"], dtype=float)
bottom_left = np.array(measured_points["bottom_left"], dtype=float)
bottom_right = np.array(measured_points["bottom_right"], dtype=float)

x_top = (top_left[0] + top_right[0]) / 2
x_bottom = (bottom_left[0] + bottom_right[0]) / 2
y_left = (top_left[1] + bottom_left[1]) / 2
y_right = (top_right[1] + bottom_right[1]) / 2
z_place = 0.03

x_middle = (x_top + x_bottom) / 2
y_middle = (y_left + y_right) / 2

table_named_points = {
    "top left corner": np.array([x_top, y_left, z_place], dtype=float),
    "top side": np.array([x_top, y_middle, z_place], dtype=float),
    "top right corner": np.array([x_top, y_right, z_place], dtype=float),

    "left side": np.array([x_middle, y_left, z_place], dtype=float),
    "middle": np.array([x_middle, y_middle, z_place], dtype=float),
    "right side": np.array([x_middle, y_right, z_place], dtype=float),

    "bottom left corner": np.array([x_bottom, y_left, z_place], dtype=float),
    "bottom side": np.array([x_bottom, y_middle, z_place], dtype=float),
    "bottom right corner": np.array([x_bottom, y_right, z_place], dtype=float),
}

for k, v in table_named_points.items():
    print(k, "=", np.round(v, 4).tolist())

top left corner = [0.35, 0.13, 0.03]
top side = [0.35, -0.0, 0.03]
top right corner = [0.35, -0.13, 0.03]
left side = [0.25, 0.13, 0.03]
middle = [0.25, -0.0, 0.03]
right side = [0.25, -0.13, 0.03]
bottom left corner = [0.15, 0.13, 0.03]
bottom side = [0.15, -0.0, 0.03]
bottom right corner = [0.15, -0.13, 0.03]


In [67]:
print(env.get_obj_pos('top left corner'))
print(env.get_obj_pos('top right corner'))
print(env.get_obj_pos('bottom left corner'))
print(env.get_obj_pos('bottom right corner'))
print(env.get_obj_pos('left side'))
print(env.get_obj_pos('right side'))
print(env.get_obj_pos('top side'))
print(env.get_obj_pos('bottom side'))

[0.35 0.13 0.02]
[ 0.35 -0.13  0.02]
[0.15 0.13 0.02]
[ 0.15 -0.13  0.02]
[0.25 0.13 0.02]
[ 0.25 -0.13  0.02]
[0.35 0.   0.02]
[0.15 0.   0.02]


In [1]:
print(hasattr(env, "_resolve_place_xyz"))

NameError: name 'env' is not defined